In [1]:
import glob
import pandas as pd
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Load all dataset parts (sorted for reproducibility)
files = sorted(glob.glob("../data/raw/uwb_dataset_part*.csv"))
print("Files found:", len(files))

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Dataset shape:", df.shape)
df.head()

Files found: 7
Dataset shape: (42000, 1031)


,NLOS,RANGE,FP_IDX,FP_AMP1,FP_AMP2,FP_AMP3,STDEV_NOISE,CIR_PWR,MAX_NOISE,RXPACC,...,CIR1006,CIR1007,CIR1008,CIR1009,CIR1010,CIR1011,CIR1012,CIR1013,CIR1014,CIR1015
0,0.0,3.90,745.0,18712.0,10250.0,11576.0,64.0,11855.0,967.0,611.0,...,279.0,458.0,183.0,158.0,198.0,87.0,296.0,505.0,307.0,0.0
1,0.0,0.66,749.0,11239.0,6313.0,4712.0,64.0,18968.0,1133.0,447.0,...,144.0,334.0,290.0,228.0,187.0,213.0,202.0,89.0,103.0,0.0
2,1.0,7.86,746.0,4355.0,5240.0,3478.0,60.0,14699.0,894.0,723.0,...,32.0,373.0,224.0,174.0,124.0,329.0,207.0,96.0,218.0,0.0
3,1.0,3.48,750.0,8502.0,8416.0,5890.0,76.0,8748.0,1127.0,1024.0,...,252.0,173.0,198.0,160.0,434.0,397.0,290.0,155.0,342.0,256.0
4,0.0,1.19,746.0,17845.0,18095.0,12058.0,68.0,11380.0,1744.0,276.0,...,154.0,209.0,242.0,296.0,87.0,178.0,314.0,247.0,292.0,256.0


In [2]:
# Select waveform columns automatically
CIR_COLUMNS = [col for col in df.columns if col.startswith("CIR")]

print("Number of CIR columns:", len(CIR_COLUMNS))
print("First 5 CIR columns:", CIR_COLUMNS[:5])
print("Last 5 CIR columns:", CIR_COLUMNS[-5:])

X_raw = df[CIR_COLUMNS].copy()

Number of CIR columns: 1017
First 5 CIR columns: ['CIR_PWR', 'CIR0', 'CIR1', 'CIR2', 'CIR3']
Last 5 CIR columns: ['CIR1011', 'CIR1012', 'CIR1013', 'CIR1014', 'CIR1015']


In [3]:
idx_path = Path("../data/processed")

train_idx = pd.read_csv(idx_path / "train_idx.csv")["idx"].to_numpy()
test_idx  = pd.read_csv(idx_path / "test_idx.csv")["idx"].to_numpy()

X_train_raw = X_raw.loc[train_idx].reset_index(drop=True)
X_test_raw  = X_raw.loc[test_idx].reset_index(drop=True)

print("Train raw shape:", X_train_raw.shape)
print("Test raw shape :", X_test_raw.shape)

Train raw shape: (33600, 1017)
Test raw shape : (8400, 1017)


In [4]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print("Scaling complete.")

Scaling complete.


In [5]:
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print("Number of components selected:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

Number of components selected: 860
Total explained variance: 0.9500271868845315


In [6]:
out = Path("../data/processed/pca")
out.mkdir(parents=True, exist_ok=True)

# Convert to DataFrame with readable column names
X_train_pca_df = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i+1}" for i in range(X_train_pca.shape[1])]
)

X_test_pca_df = pd.DataFrame(
    X_test_pca,
    columns=[f"PC{i+1}" for i in range(X_test_pca.shape[1])]
)

X_train_pca_df.to_csv(out / "X_train_pca.csv", index=False)
X_test_pca_df.to_csv(out / "X_test_pca.csv", index=False)

print("Saved PCA files:")
print(out / "X_train_pca.csv")
print(out / "X_test_pca.csv")

Saved PCA files:
..\data\processed\pca\X_train_pca.csv
..\data\processed\pca\X_test_pca.csv
